# Core-safety check on a virtual UniMMCore

This notebook runs the fixes of `fix/core-safety` end to end on a real pymmcore-plus `UniMMCore`: the [virtual-microscope-teaching](https://github.com/hinderling/virtual-microscope-teaching) scope behind `UniMMCoreSimulation`, in realtime mode, with napari-micromanager and the status widget open. The test suite uses a fake microscope; here the frames go through the real MDA runner, the real DMD path and the Qt GUI.

This is a manual check, not collected by pytest. Open it in Jupyter and run all cells. Every check is an `assert`: if the last cell prints, every check passed. Set `OPEN_NAPARI = False` to run it without a GUI.

| Section | What runs | Fixes it checks |
|---|---|---|
| 2. Guards | `Controller` properties and a run without storage | c48d420, c928529 |
| 3. Run A | baseline with an inline `mVenus` ref frame, stimulation, recovery, ref phase; napari open | c8d2e0c, fb08bf0, 57e3913, 9c9fbc0, 0cdab83 |
| 4. Run B | `continue_experiment` with a pipeline queue of 1, so frames (ref frames too) are deferred and reloaded from storage | 61e7cd9, aeb42f2, 57e3913, e373069 |
| 5. Reanalysis of A+B | OME-Zarr source with a stim channel, inline ref frames and a ref phase | c3d892c, a6ba2e2, 9ba06d3 |
| 6. Run C (TIFF) | TIFF reanalysis, `OmeZarrWriter` on a TIFF source, replay with `ControllerSimulated` | b0f63b1, 4ea5e72, 8f4a32a |
| 7. Run D (plate, 1 channel) | replay of a one-channel `OmeZarrWriterPlate` store | e05f663 |

Not checked here, because they need broken or old data (the unit tests cover them): f464b1b, 84a9af7 and 6b73475 (reanalysis of old data, a missing `ref/` image, a FOV without cells), abc2395 (a full storage queue), 499a9f5 (a ref frame without its ref channels) and cd51941 (a `df_acquire` read from parquet). A lost frame would still fail the background-error and timestep checks below.

In [ ]:
INTERVAL_S = 1.0  # seconds between frames
N_BASELINE = 5  # frames before stimulation; the first one also images REF_CHANNEL
N_STIM = 20  # frames with stimulation
N_RECOVERY = 5  # frames after stimulation, followed by one ref phase frame
N_BURST = 20  # run B: frames at interval 0, every second one with REF_CHANNEL
N_SMALL = 4  # runs C and D: timepoints
STIM_FRACTION = 0.2  # illuminate the top 20 % of each cell
STIM_CHANNEL = {"config": "CyanStim", "exposure": 50}
REF_CHANNEL = {"config": "mVenus", "exposure": 50}  # optoFGFR, the reference
IMAGING = [
    {"config": "phase-contrast", "exposure": 50},
    {"config": "miRFP", "exposure": 50},  # nuclei, segmented
]
OPEN_NAPARI = True  # the Qt fix (9c9fbc0) is only exercised with napari open

## 1. Microscope, napari and components

`core` is a `UniMMCore`. The components are those of [live_experiment.ipynb](../examples/live_experiment.ipynb), plus a reference feature extractor: `RefFE` measures a one-channel image, so `MVenusRefFE` gives it the last channel. That is the `mVenus` ref channel of an inline ref frame and the last channel of the ref phase.

In [ ]:
import os
import tempfile
import threading
import time

import numpy as np
import pandas as pd
from vmteach import load_microscope

from faro.microscope.simulation import UniMMCoreSimulation

core, sim = load_microscope("optogenetic", mode="realtime")
mic = UniMMCoreSimulation(mmc=core)
mic.init_scope()
mic.calibrate_dmd(STIM_CHANNEL)
ROOT = tempfile.mkdtemp(prefix="faro_core_safety_")
print(f"{type(core).__mro__[1].__name__}, results in {ROOT}")

In [ ]:
if OPEN_NAPARI:
    import napari
    from napari_micromanager import MainWindow
    from faro.widgets import ExperimentStatusWidget

    viewer = napari.Viewer()
    viewer.window.add_dock_widget(
        MainWindow(viewer, mmcore=core), name="napari-micromanager"
    )
    status_widget = ExperimentStatusWidget()
    viewer.window.add_dock_widget(status_widget, name="experiment status", area="right")

In [ ]:
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.filters import gaussian, threshold_otsu
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import dilation, disk
from skimage.segmentation import expand_labels, watershed

from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.base import FeatureExtractor
from faro.feature_extraction.ref import RefFE
from faro.segmentation.base import Segmentator
from faro.stimulation.base import StimWithPipeline
from faro.tracking.trackpy import TrackerTrackpy


class WatershedSegmentator(Segmentator):
    """Blur, Otsu threshold, distance transform, watershed; grow nuclei to cells."""

    def segment(self, image):
        blurred = gaussian(image, sigma=3)
        binary = blurred > threshold_otsu(blurred)
        distance = ndimage.distance_transform_edt(binary)
        coords = peak_local_max(distance, min_distance=20, labels=binary)
        markers = np.zeros(distance.shape, dtype=bool)
        markers[tuple(coords.T)] = True
        nuclei = watershed(-distance, label(markers), mask=binary)
        return expand_labels(nuclei, distance=12)


class AreaAndIntensity(FeatureExtractor):
    def __init__(self, used_mask):
        self.used_mask = used_mask
        super().__init__()

    def extract_features(self, labels, image, df_tracked=None, metadata=None):
        table = regionprops_table(
            labels[self.used_mask],
            intensity_image=image[0],
            properties=["label", "area", "mean_intensity"],
        )
        return pd.DataFrame(table), None


class StimUp(StimWithPipeline):
    """Illuminate the top `fraction` of each cell, dilated by 3 px."""

    def __init__(self, fraction):
        self.fraction = fraction

    def get_stim_mask(self, label_images, metadata=None, img=None, tracks=None):
        labels = label_images["labels"]
        stim_mask = np.zeros(labels.shape, dtype=np.uint8)
        for prop in regionprops(labels):
            minr, _, maxr, _ = prop.bbox
            rows, cols = np.where(labels == prop.label)
            top = rows < minr + self.fraction * (maxr - minr)
            local = np.zeros_like(stim_mask)
            local[rows[top], cols[top]] = 1
            stim_mask = np.maximum(stim_mask, dilation(local, footprint=disk(3)))
        return stim_mask, None


class MVenusRefFE(RefFE):
    """RefFE on the last channel of the reference image."""

    def extract_features(self, segmentation_results, image, df_tracked, metadata):
        return super().extract_features(
            segmentation_results, image[-1:], df_tracked, metadata
        )


def components(seg_channel=1):
    """Pipeline components; seg_channel indexes the imaging channels."""
    return dict(
        segmentators=[
            SegmentationMethod("labels", WatershedSegmentator(), seg_channel, True)
        ],
        feature_extractor=AreaAndIntensity("labels"),
        tracker=TrackerTrackpy(search_range=15),
        stimulator=StimUp(STIM_FRACTION),
        feature_extractor_ref=MVenusRefFE("labels"),
    )


def make_pipeline(path, **kwargs):
    return ImageProcessingPipeline(storage_path=path, **components(**kwargs))

### Checks

`check_run` checks a finished run: no background errors, every imaging and ref frame received exactly once (a frame callback left connected from an earlier run would count frames twice, 57e3913), the frame callbacks are disconnected again, and the MDA thread has ended (it hangs at exit when Qt events stay on it, 9c9fbc0). `check_tracks` checks the tracks: every timestep once per cell, and reference features on every cell of every ref frame, particle 0 included (fb08bf0).

In [ ]:
from faro.core.data_structures import Channel, ImgType, RTMSequence, combine
from faro.core.conversion import load_events_json
from faro.core.qt import pump_qt_events


def expect_raises(exc, fn):
    try:
        fn()
    except exc as e:
        print(f"ok, {type(e).__name__}: {e}")
        return
    raise AssertionError(f"expected {exc.__name__}")


def n_frames(events):
    """Imaging and ref frames of the events; stim frames do not count."""
    return sum(len(e.channels) + len(e.ref_channels) for e in events)


def ref_timesteps(events):
    return sorted(
        e.index["t"]
        for e in events
        if e.ref_channels or e.metadata.get("img_type") == ImgType.IMG_REF
    )


def n_frame_callbacks():
    return len(core.mda.events.frameReady)


def run_and_check(handle, events, n_callbacks):
    """Wait for the run, keeping napari responsive, and check it."""
    while handle.is_running():
        pump_qt_events()
        time.sleep(0.05)
    status = handle.wait()
    assert status.state == "done", (status.state, status.fatal_error)
    assert not status.background_errors, status.background_errors
    assert status.n_frames_received == n_frames(
        events
    ), f"{status.n_frames_received} frames received, {n_frames(events)} expected"
    assert status.n_stim_fallbacks == 0, f"{status.n_stim_fallbacks} stim fallbacks"
    assert n_frame_callbacks() == n_callbacks, "a frame callback stayed connected"
    alive = [t.name for t in threading.enumerate() if "_run_and_drain" in t.name]
    assert not alive, f"MDA thread still alive: {alive}"
    print(f"run ok: {status.n_frames_received} frames, {len(events)} events")
    return status


def read_tracks(path, fov=0):
    return pd.read_parquet(os.path.join(path, "tracks", f"{fov}_latest.parquet"))


def check_tracks(tracks, events):
    assert sorted(tracks["timestep"].unique()) == sorted(
        {e.index["t"] for e in events}
    ), "a timestep is missing or extra"
    assert not tracks.duplicated(["particle", "timestep"]).any(), "a frame ran twice"
    ref_ts = ref_timesteps(events)
    at_ref = tracks[tracks["timestep"].isin(ref_ts)]
    missing = sorted(at_ref[at_ref["ref_mean_intensity"].isna()]["timestep"].unique())
    assert not missing, f"ref features missing at timesteps {missing}"
    assert 0 in at_ref["particle"].values, "particle 0 has no ref frame"
    others = tracks[~tracks["timestep"].isin(ref_ts)]
    assert others["ref_mean_intensity"].isna().all(), "ref features on a non-ref frame"
    print(f"tracks ok: {tracks['particle'].nunique()} particles, ref frames {ref_ts}")


def check_same_ref_features(a, b, events):
    """Same reference intensities at every ref frame.

    Segmentation is deterministic, so the labels are the same; the particle
    ids need not be.
    """
    for t in ref_timesteps(events):
        va = np.sort(a.loc[a["timestep"] == t, "ref_mean_intensity"].to_numpy())
        vb = np.sort(b.loc[b["timestep"] == t, "ref_mean_intensity"].to_numpy())
        assert va.shape == vb.shape and np.allclose(va, vb), f"differ at t={t}"
    print(f"same ref features at {ref_timesteps(events)}")

## 2. Guards

A `Controller` keeps its pipeline and writer (c48d420), and refuses a run that would store nothing (c928529).

In [ ]:
from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter

path_a = os.path.join(ROOT, "run_a")
ctrl = Controller(mic, make_pipeline(path_a), writer=OmeZarrWriter(path_a))
expect_raises(AttributeError, lambda: setattr(ctrl, "pipeline", None))
expect_raises(AttributeError, lambda: setattr(ctrl, "writer", None))
expect_raises(RuntimeError, lambda: Controller(mic, None).run_experiment([]))

## 3. Run A: inline ref frame, stimulation, ref phase

The baseline images `mVenus` as a ref channel on its first frame. After the recovery, a ref phase (`img_type=IMG_REF`, no `ref_channels`) images all three channels once; its imaging channels are the reference (c8d2e0c). `events.json` must keep that `img_type` (0cdab83).

In [ ]:
position = [{"x": 0.0, "y": 0.0, "z": 0.0}]
ref = (Channel(**REF_CHANNEL),)

events_a = combine(
    RTMSequence(
        time_plan={"interval": INTERVAL_S, "loops": N_BASELINE},
        stage_positions=position,
        channels=IMAGING,
        ref_channels=ref,
        ref_frames={0},
        rtm_metadata={"phase": "baseline"},
    ),
    RTMSequence(
        time_plan={"interval": INTERVAL_S, "loops": N_STIM},
        stage_positions=position,
        channels=IMAGING,
        stim_channels=[STIM_CHANNEL],
        stim_frames=range(N_STIM),
        rtm_metadata={"phase": "stimulation"},
    ),
    RTMSequence(
        time_plan={"interval": INTERVAL_S, "loops": N_RECOVERY},
        stage_positions=position,
        channels=IMAGING,
        rtm_metadata={"phase": "recovery"},
    ),
    RTMSequence(
        time_plan={"interval": 0, "loops": 1},
        stage_positions=position,
        channels=IMAGING + [REF_CHANNEL],
        rtm_metadata={"phase": "ref", "img_type": ImgType.IMG_REF},
    ),
    axis="t",
)
assert ctrl.validate_events(events_a)
if OPEN_NAPARI:
    status_widget.set_controller(ctrl)

n_cb = n_frame_callbacks()
status_a = run_and_check(
    ctrl.run_experiment(events_a, stim_mode="current"), events_a, n_cb
)
saved = load_events_json(path_a)
assert saved[-1].metadata["img_type"] is ImgType.IMG_REF, saved[-1].metadata
print("events.json keeps img_type")

## 4. Run B: deferred frames

`continue_experiment` keeps the Analyzer. Its pipeline queue is lowered to 1 (a private attribute; there is no public setting), so at interval 0 most frames are deferred: the Analyzer drops them from RAM and reloads them from storage when the pipeline is free. A deferred ref frame must come back with its ref channel (61e7cd9), deferred frames must reach the pipeline in order, so no timestep is lost (e373069), and `finish_experiment` must wait until every deferred frame is processed and stored (aeb42f2).

In [ ]:
ctrl._analyzer.max_queue_size = 1
burst = list(
    RTMSequence(
        time_plan={"interval": 0, "loops": N_BURST},
        stage_positions=position,
        channels=IMAGING,
        ref_channels=ref,
        ref_frames=range(0, N_BURST, 2),
        rtm_metadata={"phase": "burst"},
    )
)
n_cb = n_frame_callbacks()
status_b = run_and_check(
    ctrl.continue_experiment(burst, stim_mode="current"), burst, n_cb
)
analyzer = ctrl._analyzer  # finish_experiment drops it; keep it for its counters
ctrl.finish_experiment()
stats = analyzer.queue_stats()
print(stats)
assert stats.skipped_pipeline > 0, "no frame was deferred"
assert stats.deferred_processed == stats.skipped_pipeline, "a deferred frame was lost"

events_ab = load_events_json(path_a)
tracks_a = read_tracks(path_a)
check_tracks(tracks_a, events_ab)

Did the cells respond? They should move up (to smaller row index, column `x`) while the light is on.

In [ ]:
import matplotlib.pyplot as plt

tracks = tracks_a.sort_values(["particle", "timestep"])
tracks["cum_dy"] = -tracks.groupby("particle")["x"].diff().fillna(0)
tracks["cum_dy"] = tracks.groupby("particle")["cum_dy"].cumsum()
displacement = tracks.groupby("timestep")["cum_dy"].median()
stim_t = [e.index["t"] for e in events_ab if e.stim_channels]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axvspan(min(stim_t), max(stim_t), color="cyan", alpha=0.15, label="stimulation")
ax.plot(displacement.index, displacement.values, color="k", lw=2, label="median")
ax.set_xlabel("timestep")
ax.set_ylabel("upward displacement (px)")
ax.legend()
print(
    f"median upward displacement during stimulation: {displacement[max(stim_t)] - displacement[min(stim_t)]:.1f} px"
)
assert displacement[max(stim_t)] > displacement[min(stim_t)], "cells did not move up"

## 5. Reanalysis of runs A and B

The OME-Zarr store holds a stim channel, which the reanalysis must strip (9ba06d3). The inline ref frames read their ref channel from `ref/` (a6ba2e2), the ref phase frame its whole stack (c3d892c). The reanalysis must measure the same reference intensities as the live run.

In [ ]:
from faro.core.pipeline_post import ImageProcessingPipeline_postExperiment
from faro.core.writers import OmeZarrWriterPlate, TiffWriter


def reanalyse(src, writer_cls, **kwargs):
    out = src + "_reanalysis"
    ImageProcessingPipeline_postExperiment(
        img_storage_path=src,
        out_path=out,
        events=load_events_json(src),
        n_jobs=1,
        writer=writer_cls(out),
        **components(**kwargs),
    ).run()
    return out


tracks_a_post = read_tracks(reanalyse(path_a, OmeZarrWriter))
check_tracks(tracks_a_post, events_ab)
check_same_ref_features(tracks_a, tracks_a_post, events_ab)

## 6. Run C: TIFF storage

A short run with a `TiffWriter`: an inline ref frame and a ref phase. An `OmeZarrWriter` reanalysis of a TIFF source must fail at once with a clear message (4ea5e72); a `TiffWriter` reanalysis works (b0f63b1). `ControllerSimulated` replays the stored frames, ref frames rebuilt as `[imaging | ref]` (8f4a32a).

In [ ]:
from faro.core.controller import ControllerSimulated


def replay(src, **kwargs):
    out = src + "_replay"
    events = load_events_json(src)
    ctrl_r = ControllerSimulated(
        mic, make_pipeline(out, **kwargs), src, writer=TiffWriter(out)
    )
    n_cb = n_frame_callbacks()
    run_and_check(ctrl_r.run_experiment(events, validate=False), events, n_cb)
    ctrl_r.finish_experiment()
    return out


path_c = os.path.join(ROOT, "run_c_tiff")
events_c = combine(
    RTMSequence(
        time_plan={"interval": INTERVAL_S, "loops": N_SMALL},
        stage_positions=position,
        channels=IMAGING,
        ref_channels=ref,
        ref_frames={1},
    ),
    RTMSequence(
        time_plan={"interval": 0, "loops": 1},
        stage_positions=position,
        channels=IMAGING + [REF_CHANNEL],
        rtm_metadata={"img_type": ImgType.IMG_REF},
    ),
    axis="t",
)
ctrl_c = Controller(mic, make_pipeline(path_c), writer=TiffWriter(path_c))
if OPEN_NAPARI:
    status_widget.set_controller(ctrl_c)
n_cb = n_frame_callbacks()
run_and_check(ctrl_c.run_experiment(events_c), events_c, n_cb)
ctrl_c.finish_experiment()
tracks_c = read_tracks(path_c)
check_tracks(tracks_c, events_c)

expect_raises(
    ValueError,
    lambda: ImageProcessingPipeline_postExperiment(
        img_storage_path=path_c,
        out_path=path_c + "_wrong",
        events=load_events_json(path_c),
        writer=OmeZarrWriter(path_c + "_wrong"),
        **components(),
    ),
)
check_same_ref_features(tracks_c, read_tracks(reanalyse(path_c, TiffWriter)), events_c)
check_same_ref_features(tracks_c, read_tracks(replay(path_c)), events_c)

## 7. Run D: one-channel plate

An `OmeZarrWriterPlate` store with only `miRFP` has well arrays without a channel axis; reading a frame back must still give `(c, y, x)` (e05f663). The replay must see the same images as the live run, so it finds the same cells.

In [ ]:
path_d = os.path.join(ROOT, "run_d_plate")
events_d = list(
    RTMSequence(
        time_plan={"interval": INTERVAL_S, "loops": N_SMALL},
        stage_positions=[
            {"x": 0.0, "y": 0.0, "z": 0.0},
            {"x": 600.0, "y": 0.0, "z": 0.0},
        ],
        channels=[IMAGING[1]],
    )
)
ctrl_d = Controller(
    mic, make_pipeline(path_d, seg_channel=0), writer=OmeZarrWriterPlate(path_d)
)
if OPEN_NAPARI:
    status_widget.set_controller(ctrl_d)
n_cb = n_frame_callbacks()
run_and_check(ctrl_d.run_experiment(events_d), events_d, n_cb)
ctrl_d.finish_experiment()

replay_d = replay(path_d, seg_channel=0)
for fov in (0, 1):
    live, again = read_tracks(path_d, fov), read_tracks(replay_d, fov)
    assert np.array_equal(np.sort(live["area"]), np.sort(again["area"])), fov
    print(f"fov {fov}: replay finds the same {len(live)} cells")

In [ ]:
print("All checks passed. Results in", ROOT)